# 数値計算演習C：軸対称・非軸対称ポテンシャル中の軌道［学生用・コード入力版］

**Ryden, *Celestial and Stellar Dynamics*, §7.2–7.3**

Notebook 3のリープフロッグ法を、異なる重力場へ再利用します。加速度関数を差し替えるだけでなく、各モデルで使える保存量と比較の基準を選び直し、数値軌道を検証します。解析的な加速度を使うため、Notebook BのPoisson解は必要ありません。

## 学習目標

1. 加速度関数と初期条件を設定し、第3章の時間積分法を異なる重力場へ再利用できる。
2. ポテンシャルの対称性に応じて保存量を選び、数値軌道の診断に用いることができる。
3. 厳密解が得られない問題で、小振幅近似を近似的な参照解として用い、数値軌道との差を評価できる。
4. 時間刻みと初期振幅を別々に変え、時間積分の誤差と小振幅近似との差を区別できる。
5. 非軸対称系の角運動量の変化を数値誤差と区別し、有限時間の軌道から判断できる範囲を説明できる。

## 学生用：コード入力版の使い方

テキスト『銀河力学で学ぶ数値計算法』v1.2 を開き、**「入力」セルに指定された掲載コードを書き写してから、上から順に実行**します。「入力済み」セルは、準備・描画・結果表示などのコードです。そのまま実行してください。

入力セルにある `raise NotImplementedError(...)` は、未入力で先へ進まないための行です。この1行を削除してコードを入力します。入力前に実行してこのエラーが出るのは正常です。関数や `for`・`if` の内側では、テキストと同じ字下げを保ってください。

コードを実行するには **Shift + Enter** を押します。入力後は保存し、最後にカーネルを再起動して全セルを上から実行すると、入力漏れを確認できます。別の Notebook を先に実行する必要はありません。

ページ番号は v1.2 に対応しています。`p.` は本文に印刷されたページ番号、`PDF` は表紙から数えたページ番号です。同じ計算が短い Listing と完全な関数の両方にある場合は、完全な関数の箇所で一度だけ入力します。

## 入力箇所一覧

| 入力欄 | 内容 | テキスト |
| --- | --- | --- |
| [C-01](#input-C-01) | 対数ポテンシャルの加速度 | p.76 |
| [C-02](#input-C-02) | エピサイクル運動の初期条件 | p.78 |
| [C-03](#input-C-03) | 軌道積分と診断量 | p.78 |
| [C-04](#input-C-04) | 時間刻みを変えた軌道計算 | p.81 |
| [C-05](#input-C-05) | 共通時刻での位置の差と収束次数 | p.80 |
| [C-06](#input-C-06) | 初期振幅を変えた比較 | p.81 |
| [C-07](#input-C-07) | Miyamoto–Nagai ポテンシャルの加速度 | p.82 |
| [C-08](#input-C-08) | Miyamoto–Nagai 軌道の計算 | p.83 |
| [C-09](#input-C-09) | 非軸対称ポテンシャルの加速度 | p.84 |
| [C-10](#input-C-10) | 同じエネルギーを持つ初期条件 | p.85 |
| [C-11](#input-C-11) | 2本の軌道の計算と診断量 | p.85 |

In [ ]:
# 入力済み：このセルはそのまま実行する。
import numpy as np
import matplotlib.pyplot as plt

G = 1.0

OI = {
    "orange": "#E69F00",
    "sky": "#56B4E9",
    "green": "#009E73",
    "yellow": "#F0E442",
    "blue": "#0072B2",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "black": "#000000",
}

plt.rcParams["figure.figsize"] = (6.5, 4.5)
plt.rcParams["font.size"] = 11


In [ ]:
# 入力済み：このセルはそのまま実行する。
def integrate_leapfrog(acc_func, x0, v0, dt, nstep):
    """KDK形式のリープフロッグ法。更新式はNotebook 3と同じ。"""
    x0 = np.asarray(x0, dtype=float)
    v0 = np.asarray(v0, dtype=float)
    ndim = len(x0)
    x = np.empty((nstep + 1, ndim))
    v = np.empty((nstep + 1, ndim))
    x[0], v[0] = x0, v0

    for n in range(nstep):
        # Kick: 現在位置の加速度で速度を半ステップ進める。
        v_half = v[n] + 0.5 * dt * acc_func(x[n])
        # Drift: 半ステップの速度で位置を1ステップ進める。
        x[n + 1] = x[n] + dt * v_half
        # Kick: 新しい位置の加速度で速度を残り半ステップ進める。
        v[n + 1] = v_half + 0.5 * dt * acc_func(x[n + 1])

    return x, v


In [ ]:
# 入力済み：このセルはそのまま実行する。
def angular_momentum_z(x, v):
    """2次元・3次元の軌道配列から、各時刻のLzを求める。"""
    return x[:, 0] * v[:, 1] - x[:, 1] * v[:, 0]


# 1. 軸対称ポテンシャル

Ryden §7.2の軸対称系を題材に、時間に依存しない対数ポテンシャルを使います。
$$
\Phi(R,z)=\frac12v_0^2\ln\left(\frac{R^2+z^2/q^2}{\ell^2}\right).
$$
$v_0>0$は速度のスケール、$q>0$は等ポテンシャル面の形を決める定数です。$q<1$では扁平な形になります。$\ell$は対数の引数を無次元にする固定の基準長さで、以下のコードでは$\ell=1$とします。これはポテンシャルの加法定数の選択であり、加速度を変えません。原点ではポテンシャルと加速度が特異になるため、原点を通る軌道にはこのモデルを使いません。

$D=x^2+y^2+z^2/q^2$と置き、$\boldsymbol a=-\nabla\Phi$を各座標で計算すると、
$$
\boldsymbol a=-\frac{v_0^2}{D}(x,y,z/q^2)
$$
です。`acc_log_axisym`がこの式に対応し、積分器へ渡すのはこの加速度関数です。

軸対称性から$z$軸まわりのトルクが0なので、$L_z$が保存します。Rydenの記号$j_z$に対応します。半径・鉛直方向の運動を記述する有効ポテンシャルは
$$
\Phi_{\rm eff}(R,z)=\Phi(R,z)+\frac{L_z^2}{2R^2}
$$
です（Ryden式7.35）。$L_z$を固定し、この関数を円軌道のまわりで展開して比較用の近似解を求めます。

**入力済み｜モデルのパラメータと診断用の関数**

In [ ]:
# 入力済み：このセルはそのまま実行する。
vlog = 1.0
qlog = 0.8

def phi_log_axisym(x, v0=1.0, q=0.8):
    """位置1点または軌道配列のポテンシャル。基準長さell=1。"""
    x = np.asarray(x)
    R2 = x[..., 0]**2 + x[..., 1]**2
    z2 = x[..., 2]**2
    return 0.5 * v0**2 * np.log(R2 + z2 / q**2)

def energy_log_axisym(x, v, v0=1.0, q=0.8):
    return 0.5 * np.sum(v**2, axis=1) + phi_log_axisym(x, v0, q)

def Lz_3d(x, v):
    return angular_momentum_z(x, v)


<a id="input-C-01"></a>
### 入力 C-01｜対数ポテンシャルの加速度

**参照：Listing C.1、テキスト v1.3 p.77（PDF 83ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=83)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-01：対数ポテンシャルの加速度
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-01 のコードを入力してください。")


## 1.1 ほぼ円軌道とエピサイクル運動

円軌道の案内中心半径を$R_g>0$とし、正の向きに回る軌道を考えます。$\partial\Phi_{\rm eff}/\partial R=0$から$L_z=R_gv_0$です。半径方向の変位を$\xi=R-R_g$とすると、
$$
\Omega=\frac{v_0}{R_g},\qquad
\kappa^2=\left.\frac{\partial^2\Phi_{\rm eff}}{\partial R^2}\right|_{(R_g,0)}=\frac{2v_0^2}{R_g^2},\qquad
\nu_z^2=\left.\frac{\partial^2\Phi}{\partial z^2}\right|_{(R_g,0)}=\frac{v_0^2}{q^2R_g^2}.
$$
$\kappa$は半径方向のエピサイクル振動数、$\nu_z$は鉛直振動数です（Ryden式7.47）。$\xi_0/R_g\ll1$、$z_0/(qR_g)\ll1$で、初期の半径・鉛直速度を0とすると、線形化した運動方程式の解は
$$
R(t)-R_g\simeq\xi_0\cos(\kappa t),\qquad z(t)\simeq z_0\cos(\nu_z t)
$$
となります。これは元の運動方程式の厳密解ではなく、小振幅で成り立つ近似解です。

`x_amp`が$\xi_0$、`z_amp`が$z_0$に対応します。両者を$0.05R_g$とし、$R_{\rm init}=R_g+\xi_0$から始めます。初期の方位角速度成分を$v_{\phi,\rm init}=R_gv_0/R_{\rm init}$にして、$L_z=R_gv_0$を保ちます。

$T_\phi=2\pi/\Omega$を基準周期とし、$\Delta t=T_\phi/300$で$12T_\phi$まで積分します。振動の図には最初の$3T_\phi$を示します。$12T_\phi$は案内中心の円軌道の周期で表した積分時間であり、数値軌道がちょうど12周するという意味ではありません。

保存量の最大絶対相対変化は、全出力時刻で$\max|E(t_n)-E(0)|/|E(0)|$と$\max|L_z(t_n)-L_z(0)|/|L_z(0)|$を測ります。以下の例では両分母は0ではありません。ポテンシャルの加法定数を変えると$E(0)$も変わるので、異なる規格化のエネルギー相対誤差をそのまま比較しないでください。

<a id="input-C-02"></a>
### 入力 C-02｜エピサイクル運動の初期条件

**参照：§C.2「ほぼ円軌道とエピサイクル運動」対応セル（初期条件と基準周期）、テキスト v1.3 p.79（PDF 85ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=85)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-02：エピサイクル運動の初期条件
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-02 のコードを入力してください。")


<a id="input-C-03"></a>
### 入力 C-03｜軌道積分と診断量

**参照：§C.2「ほぼ円軌道とエピサイクル運動」対応セル（時間積分と診断量）、テキスト v1.3 p.79（PDF 85ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=85)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-03：軌道積分と診断量
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-03 のコードを入力してください。")


**入力済み｜振動数と保存量の表示**

In [ ]:
# 入力済み：このセルはそのまま実行する。
print(f"Omega   = {Omega:.6f}")
print(f"kappa   = {kappa:.6f}")
print(f"nu_z    = {nu_z:.6f}")
print(f"最大 |Delta E/E0|   = {np.max(np.abs(E_ax-E_ax[0]))/abs(E_ax[0]):.3e}")
print(f"最大 |Delta Lz/Lz0| = {np.max(np.abs(Lz_ax-Lz_ax[0]))/abs(Lz_ax[0]):.3e}")


In [ ]:
# 入力済み：このセルはそのまま実行する。
fig, ax = plt.subplots()
ax.plot(R_ax / Rg, x_ax[:, 2] / Rg, color=OI["blue"])
ax.set_xlabel(r"$R/R_g$")
ax.set_ylabel(r"$z/R_g$")
ax.set_title("Orbit in an axisymmetric logarithmic potential")
ax.grid(alpha=0.25)
plt.show()

t_show = t_ax <= 3.0 * Tphi

fig, ax = plt.subplots()
ax.plot(t_ax[t_show] / Tphi, (R_ax[t_show] - Rg) / Rg,
        color=OI["blue"], label="Numerical")
ax.plot(t_ax[t_show] / Tphi, (x_amp / Rg) * np.cos(kappa * t_ax[t_show]),
        "--", color=OI["black"], label="Epicyclic approximation")
ax.set_xlabel(r"$t/T_\phi$")
ax.set_ylabel(r"$(R-R_g)/R_g$")
ax.set_title("Radial epicycle")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots()
ax.plot(t_ax[t_show] / Tphi, x_ax[t_show, 2] / Rg,
        color=OI["vermillion"], label="Numerical")
ax.plot(t_ax[t_show] / Tphi, (z_amp / Rg) * np.cos(nu_z * t_ax[t_show]),
        "--", color=OI["black"], label="Harmonic approximation")
ax.set_xlabel(r"$t/T_\phi$")
ax.set_ylabel(r"$z/R_g$")
ax.set_title("Vertical oscillation")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


In [ ]:
# 入力済み：このセルはそのまま実行する。
fig, ax = plt.subplots()
ax.plot(t_ax / Tphi, (E_ax - E_ax[0]) / abs(E_ax[0]), color=OI["green"], label="Energy")
ax.plot(t_ax / Tphi, (Lz_ax - Lz_ax[0]) / abs(Lz_ax[0]), color=OI["purple"], label=r"$L_z$")
ax.set_xlabel(r"$t/T_\phi$")
ax.set_ylabel("Relative error")
ax.set_title(r"Conservation of $E$ and $L_z$")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 1.2 時間刻みを変えた収束試験

初期振幅$\xi_0=z_0=0.05R_g$と終端時刻$3T_\phi$を固定し、$\Delta t=T_\phi/300$から刻みを半分ずつにします。小振幅近似ではなく、同じ運動方程式を異なる時間刻みで解いた位置どうしを比較します。
$$
D_{\Delta t}=\max_{t_n}\frac{|\boldsymbol x_{\Delta t}(t_n)-\boldsymbol x_{\Delta t/2}(t_n)|}{R_g},\qquad
p_{\rm self}=\log_2\frac{D_{\Delta t}}{D_{\Delta t/2}}.
$$
比較は両方の計算に共通する時刻に限ります。細かい刻みの配列から`[::2]`で1点おきに取り出すと、粗い配列の時刻に揃います。共通の初期条件・終端時刻を使い、離散化誤差が支配的な範囲では$D_{\Delta t}\propto(\Delta t)^p$となります。$p_{\rm self}$は厳密解なしで求める収束次数の推定値であり、$D_{\Delta t}$自体は厳密解に対する誤差ではありません。

**入力済み｜時間刻みを指定して積分する補助関数**

In [ ]:
# 入力済み：このセルはそのまま実行する。
def epicycle_trial(amp_fraction, n_per_period, Rg=1.0, v0=1.0, q=0.8):
    """共通の案内中心と終端時刻3Tphiで、刻み・振幅を変える。"""
    Tphi_trial = 2.0 * np.pi * Rg / v0
    amp = amp_fraction * Rg
    x_init = np.array([Rg + amp, 0.0, amp])
    v_init = np.array([0.0, Rg * v0 / (Rg + amp), 0.0])
    nstep = 3 * int(n_per_period)
    dt = Tphi_trial / n_per_period
    acc = lambda x: acc_log_axisym(x, v0, q)
    x_trial, _ = integrate_leapfrog(acc, x_init, v_init, dt, nstep)
    t_trial = np.arange(nstep + 1) * dt
    return t_trial, x_trial


<a id="input-C-04"></a>
### 入力 C-04｜時間刻みを変えた軌道計算

**参照：§C.2「時間刻みと初期振幅の比較」対応セル（時間刻みを変えた軌道計算）、テキスト v1.3 p.82（PDF 88ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=88)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-04：時間刻みを変えた軌道計算
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-04 のコードを入力してください。")


<a id="input-C-05"></a>
### 入力 C-05｜共通時刻での位置の差と収束次数

**参照：Listing C.2、テキスト v1.3 p.81（PDF 87ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=87)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-05：共通時刻での位置の差と収束次数
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-05 のコードを入力してください。")


**入力済み｜位置の差と収束次数の表示**

In [ ]:
# 入力済み：このセルはそのまま実行する。
for npp, d in zip(steps_per_period[:-1], self_diffs):
    print(f"h/Tphi = 1/{npp:4d}, D_h = {d:.6e}")
print("隣り合う刻みでの収束次数 =", np.array2string(p_self, precision=6))


## 1.3 初期振幅を変えた近似解との比較

次に$\Delta t=T_\phi/2400$、終端時刻$3T_\phi$を固定し、共通の振幅比$\alpha=\xi_0/R_g=z_0/R_g$を$0.05,0.025,0.0125$と変えます。各振幅で$L_z=R_gv_0$となる初速度を設定し直します。
$$
A_R=\frac{\max_{t_n}|R(t_n)-R_g-\xi_0\cos(\kappa t_n)|}{|\xi_0|},\qquad
A_z=\frac{\max_{t_n}|z(t_n)-z_0\cos(\nu_z t_n)|}{|z_0|}.
$$
差は初期振幅で規格化します。これにより、振幅を小さくしたため絶対差も小さくなっただけなのか、振幅に比べても一致がよくなったのかを区別します。ここでは振幅を0にしません。

時間積分誤差が十分小さいことを確認するため、同じ比較を$\Delta t=T_\phi/1200$でも行います。両刻みで$A_R,A_z$がほぼ変わらず、振幅を小さくしたときに減るなら、小振幅近似の改善を確かめられます。振幅を極端に小さくすると固定した時間刻みでは不十分になるので、刻みの確認を繰り返してください。

<a id="input-C-06"></a>
### 入力 C-06｜初期振幅を変えた比較

**参照：§C.2「時間刻みと初期振幅の比較」対応セル（初期振幅を変えた近似解との比較）、テキスト v1.3 p.82（PDF 88ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=88)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-06：初期振幅を変えた比較
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-06 のコードを入力してください。")


**入力済み｜近似解との比較結果の表示**

In [ ]:
# 入力済み：このセルはそのまま実行する。
print("  alpha       A_R(1200) A_R(2400)   A_z(1200) A_z(2400)")
for alpha, coarse, fine in zip(amp_fractions, amp_error_coarse, amp_error_fine):
    print(f"{alpha:8.4f}      {coarse[0]:.6f}  {fine[0]:.6f}    {coarse[1]:.6f}  {fine[1]:.6f}")


# 2. Miyamoto–Nagai軌道

別の軸対称モデルへ、同じ積分器を適用します。
$$
\Phi(R,z)=-\frac{GM}{\sqrt{R^2+(a+\sqrt{z^2+b^2})^2}},\qquad G,M,a,b>0.
$$
$B=\sqrt{z^2+b^2}$、$D=\sqrt{x^2+y^2+(a+B)^2}$と置くと、
$$
g_x=-\frac{GMx}{D^3},\qquad g_y=-\frac{GMy}{D^3},\qquad g_z=-\frac{GM(a+B)z}{BD^3}
$$
です。$g_z$では$B$の$z$依存性も微分します。以下の`acc_mn_cart`がこの式に対応します。

時間に依存しない軸対称系なので、診断に使う保存量は引き続き$E$と$L_z$です。ここでは解析的なポテンシャルと加速度を使い、付録Bの格子上の数値解は入力しません。格子の数値解を使う場合には、補間や数値微分の誤差を別に調べる必要があります。

初期状態と時間は、長さ$a$、速度$v_*=\sqrt{GM/a}$、時間$t_*=a/v_*$で設定します。既定値は$G=M=a=1$、$b=0.25$、$\boldsymbol x_0/a=(3,0,0.5)$、$\boldsymbol v_0/v_*=(0,0.5,0)$、$\Delta t=0.02t_*$、終端時刻$100t_*$です。各出力時刻で保存量を測り、軌道の形だけで計算精度を判断しないでください。

**入力済み｜Miyamoto–Nagai モデルのポテンシャルとエネルギー**

In [ ]:
# 入力済み：このセルはそのまま実行する。
def phi_mn_cart(x, G=1.0, M=1.0, a=1.0, b=0.25):
    R2 = x[..., 0]**2 + x[..., 1]**2
    B = np.sqrt(x[..., 2]**2 + b**2)
    return -G * M / np.sqrt(R2 + (a + B)**2)

def energy_mn(x, v, G=1.0, M=1.0, a=1.0, b=0.25):
    return 0.5 * np.sum(v**2, axis=1) + phi_mn_cart(x, G, M, a, b)


<a id="input-C-07"></a>
### 入力 C-07｜Miyamoto–Nagai ポテンシャルの加速度

**参照：Listing C.3、テキスト v1.3 p.83（PDF 89ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=89)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-07：Miyamoto–Nagai ポテンシャルの加速度
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-07 のコードを入力してください。")


<a id="input-C-08"></a>
### 入力 C-08｜Miyamoto–Nagai 軌道の計算

**参照：§C.3「Miyamoto–Nagai 軌道」、テキスト v1.3 p.84（PDF 90ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=90)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-08：Miyamoto–Nagai 軌道の計算
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-08 のコードを入力してください。")


**入力済み｜保存量の表示と軌道の図**

In [ ]:
# 入力済み：このセルはそのまま実行する。
print(f"初期エネルギー = {E_mn[0]:.8f}")
print(f"最大 |Delta E/E0|   = {np.max(np.abs(E_mn-E_mn[0]))/abs(E_mn[0]):.3e}")
print(f"最大 |Delta Lz/Lz0| = {np.max(np.abs(Lz_mn-Lz_mn[0]))/abs(Lz_mn[0]):.3e}")

fig, ax = plt.subplots()
ax.plot(R_mn / a_mn, x_mn[:, 2] / a_mn, color=OI["blue"])
ax.set_xlabel(r"$R/a$")
ax.set_ylabel(r"$z/a$")
ax.set_title("Orbit in a Miyamoto--Nagai potential")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots()
ax.plot(x_mn[:, 0] / a_mn, x_mn[:, 1] / a_mn, color=OI["vermillion"])
ax.set_xlabel(r"$x/a$")
ax.set_ylabel(r"$y/a$")
ax.set_title("Miyamoto--Nagai orbit: face-on view")
ax.set_aspect("equal")
ax.grid(alpha=0.25)
plt.show()


# 3. 非軸対称ポテンシャル

Ryden §7.3の2次元の対数ポテンシャルを使います（式7.68）。
$$
\Phi(x,y)=\frac12v_0^2\ln\left[1+\frac{x^2+y^2/q^2}{r_c^2}\right],\qquad v_0,r_c>0,\quad 0<q\le1.
$$
$D=r_c^2+x^2+y^2/q^2$として、
$$
\boldsymbol a=-\frac{v_0^2}{D}(x,y/q^2),\qquad
\frac{dL_z}{dt}=xg_y-yg_x=\frac{v_0^2xy}{D}(1-q^{-2})
$$
です。$q\ne1$では一般にトルクが0ではなく、$L_z$は保存しません。一方、ポテンシャルは静止しているので、$E$は保存します。$L_z$の変化をすべて数値誤差と解釈しないでください。

中心近傍の$\eta=(x^2+y^2/q^2)/r_c^2\ll1$では$\ln(1+\eta)\simeq\eta$から異方的調和振動子となり、振動数は$\omegg_x=v_0/r_c$、$\omegg_y=v_0/(qr_c)$です。ただし、以下の初期位置では$\eta=4$なので、この近似を軌道全体の基準解には使いません。

Ryden図7.6にならい、$q=\sqrt3/2$、$E=1.5v_0^2$、初期位置$(2r_c,0)$を共通にします。速さは$v_{\rm init}=\sqrt{2[E-\Phi(2r_c,0)]}$、速度方向は$v_y=fv_{\rm init}$、$v_x=+\sqrt{v_{\rm init}^2-v_y^2}$で設定します。ここでは$f=0.5$と$0.9$を使います。初期位置と比エネルギーを同じにし、速度方向だけを変える比較です。

$\Delta t=0.02r_c/v_0$で$300r_c/v_0$まで積分します。この条件で、前者は回り込む向きが変わるbox軌道、後者は中心の周囲を一定の向きに回るloop軌道になります。特定の$f$の値が、別のポテンシャルや初期位置でも同じ軌道の型を与えるという意味ではありません。

図の破線は$\Phi=E$となるゼロ速度境界です。運動エネルギーが非負なので、厳密な軌道は$\Phi\le E$の領域に限られます。ただし、この領域全体を軌道が通るとは限りません。

**入力済み｜非軸対称モデルのパラメータと診断用の関数**

In [ ]:
# 入力済み：このセルはそのまま実行する。
v0_na = 1.0
rc_na = 1.0
q_na = np.sqrt(3.0) / 2.0
E_target = 1.5 * v0_na**2

def phi_nonaxisym(x, v0=1.0, rc=1.0, q=np.sqrt(3.0)/2.0):
    X, Y = np.asarray(x)[..., 0], np.asarray(x)[..., 1]
    return 0.5 * v0**2 * np.log(1.0 + (X**2 + Y**2/q**2) / rc**2)

def energy_nonaxisym(x, v, v0=1.0, rc=1.0, q=np.sqrt(3.0)/2.0):
    return 0.5 * np.sum(v**2, axis=1) + phi_nonaxisym(x, v0, rc, q)


<a id="input-C-09"></a>
### 入力 C-09｜非軸対称ポテンシャルの加速度

**参照：Listing C.4、テキスト v1.3 p.85（PDF 91ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=91)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-09：非軸対称ポテンシャルの加速度
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-09 のコードを入力してください。")


<a id="input-C-10"></a>
### 入力 C-10｜同じエネルギーを持つ初期条件

**参照：§C.4「同じエネルギーを持つ2本の軌道」対応セル（同じ比エネルギーからの初速度の設定）、テキスト v1.3 p.86（PDF 92ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=92)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-10：同じエネルギーを持つ初期条件
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-10 のコードを入力してください。")


<a id="input-C-11"></a>
### 入力 C-11｜2本の軌道の計算と診断量

**参照：§C.4「同じエネルギーを持つ2本の軌道」対応セル（時間積分と診断量）、テキスト v1.3 p.86（PDF 92ページ）**  [該当ページを開く](../../docs/galactic_dynamics_numerical_methods_v1.3.pdf#page=92)

指定したコード欄の全体を入力してください。コメントもそのまま入力できます。

次のセルの `raise NotImplementedError(...)` の行を削除し、そこに入力します。

In [ ]:
# 入力 C-11：2本の軌道の計算と診断量
# 下の raise の1行を削除し、テキストの指定箇所を入力する。
raise NotImplementedError("入力 C-11 のコードを入力してください。")


**入力済み｜エネルギーと角運動量の表示**

In [ ]:
# 入力済み：このセルはそのまま実行する。
print(f"box 軌道の初期エネルギー  = {E_box[0]:.8f}")
print(f"loop 軌道の初期エネルギー = {E_loop[0]:.8f}")
print(f"box 軌道の最大 |Delta E/E0|  = {np.max(np.abs(E_box-E_box[0]))/abs(E_box[0]):.3e}")
print(f"loop 軌道の最大 |Delta E/E0| = {np.max(np.abs(E_loop-E_loop[0]))/abs(E_loop[0]):.3e}")
print(f"box 軌道の Lz/(rc*v0) 範囲  = [{L_box.min()/(rc_na*v0_na):.4f}, {L_box.max()/(rc_na*v0_na):.4f}]")
print(f"loop 軌道の Lz/(rc*v0) 範囲 = [{L_loop.min()/(rc_na*v0_na):.4f}, {L_loop.max()/(rc_na*v0_na):.4f}]")


In [ ]:
# 入力済み：このセルはそのまま実行する。
# ゼロ速度境界 Phi(x,y)=E_target
semi_x = rc_na * np.sqrt(np.exp(2.0 * E_target / v0_na**2) - 1.0)
th = np.linspace(0.0, 2.0*np.pi, 500)
xb = semi_x * np.cos(th)
yb = q_na * semi_x * np.sin(th)

fig, ax = plt.subplots()
ax.plot(xb / rc_na, yb / rc_na, "--", color=OI["black"], label=r"$\Phi=E$")
ax.plot(x_box[:, 0] / rc_na, x_box[:, 1] / rc_na, color=OI["vermillion"], lw=0.8, label="Box orbit")
ax.set_xlabel(r"$x/r_c$")
ax.set_ylabel(r"$y/r_c$")
ax.set_title("Box orbit")
ax.set_aspect("equal")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots()
ax.plot(xb / rc_na, yb / rc_na, "--", color=OI["black"], label=r"$\Phi=E$")
ax.plot(x_loop[:, 0] / rc_na, x_loop[:, 1] / rc_na, color=OI["blue"], lw=0.8, label="Loop orbit")
ax.set_xlabel(r"$x/r_c$")
ax.set_ylabel(r"$y/r_c$")
ax.set_title("Loop orbit")
ax.set_aspect("equal")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


In [ ]:
# 入力済み：このセルはそのまま実行する。
fig, ax = plt.subplots()
ax.plot(t_na / (rc_na / v0_na), L_box / (rc_na * v0_na), color=OI["vermillion"], label="Box orbit")
ax.plot(t_na / (rc_na / v0_na), L_loop / (rc_na * v0_na), color=OI["blue"], label="Loop orbit")
ax.axhline(0.0, color=OI["black"], lw=1)
ax.set_xlabel(r"$t/(r_c/v_0)$")
ax.set_ylabel(r"$L_z/(r_c v_0)$")
ax.set_title("Angular momentum in a non-axisymmetric potential")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots()
ax.plot(t_na / (rc_na / v0_na), (E_box-E_box[0])/abs(E_box[0]), color=OI["vermillion"], label="Box orbit")
ax.plot(t_na / (rc_na / v0_na), (E_loop-E_loop[0])/abs(E_loop[0]), color=OI["blue"], label="Loop orbit")
ax.set_xlabel(r"$t/(r_c/v_0)$")
ax.set_ylabel(r"$(E-E_0)/|E_0|$")
ax.set_title("Energy conservation")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


# まとめ

数値積分器と加速度関数を分けることで、同じKDK形式のリープフロッグ法を異なる重力場へ適用できます。ただし、加速度、初期条件、診断量、時間刻みの妥当性はモデルごとに確かめ直します。

数値軌道どうしの収束と小振幅近似との一致は、異なる検証です。時間刻みを小さくしても残る差を調べるには、初期振幅も別に変える必要があります。近似解との差をすべて時間積分の誤差として扱ってはいけません。

保存量は対称性に応じて選びます。ここで扱った静止した重力場では$E$が保存します。軸対称系では$L_z$も保存しますが、非軸対称系では一般に変化します。保存量がよく保たれることだけでは、各時刻の位置の精度は保証されません。また、有限時間の出力から判断できる範囲を越えて、軌道の長時間の性質を断定しないことが重要です。